# Apply Waveform To A TPS Sequence

This notebook edits a `.tps` sequence file entirely in Python. It loads a vendored sequence, inspects parameters and table references, changes two existing parameters, applies a generated phase waveform to an `F1_PhMod` event, then writes a new `.tps` plus JSON provenance.

No TNMR COM commands are used here.

## 1. Setup

Locate the repo root and import the repo-local Redstone sequence and waveform classes.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the Hardware Capture repo root from a notebook or script working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)

from hardware_capture.rf.redstone import Sequence, Sine

## 2. Choose Input And Output Files

Edit `INPUT_TPS_PATH` to point at your own sequence when testing. The output paths are separate files under `examples/generated`, so the source sequence is not overwritten.

In [ ]:
INPUT_TPS_PATH = (
    REDSTONE_DIR
    / "references"
    / "examples"
    / "tnmr-sequences"
    / "common"
    / "1pulse - f1.tps"
)
OUTPUT_JSON_PATH = OUTPUT_DIR / "one_pulse_waveform_update.tps.json"
OUTPUT_TPS_PATH = OUTPUT_DIR / "one_pulse_waveform_update.tps"

assert INPUT_TPS_PATH.exists(), INPUT_TPS_PATH
INPUT_TPS_PATH, OUTPUT_TPS_PATH

## 3. Load And Inspect The Sequence

`Sequence.load()` decodes the TPS header, visible rows/events, table definitions, parameter pages, parameters, and preserved raw suffix chunks.

In [ ]:
sequence = Sequence.load(INPUT_TPS_PATH)
summary = sequence.summary().to_dict()
summary

## 4. Inspect Parameters And Table References

Parameter edits are intentionally exact-name updates. Table references come from event table slots in the visible sequence matrix.

In [ ]:
print("Parameters:")
for name, value in sorted(sequence.parameter_values().items()):
    print(f"  {name} = {value}")

print("\nPhase/amplitude table references:")
for reference in sequence.table_references():
    if reference.quantity in {"phase", "amplitude"}:
        print(
            f"  {reference.line} event {reference.event_number}: "
            f"{reference.table_name} ({reference.quantity})"
        )

## 5. Edit Existing Parameters

This updates only parameters that already exist in the sequence. Missing names raise an error instead of creating new parameters.

In [ ]:
sequence.set_parameter_values(
    {
        "Last Delay": "150m",
        "Acq. Time": "80m",
    }
)

sequence.parameter_values()

## 6. Apply A Phase Waveform To `F1_PhMod`

This writes a waveform into the existing `ph1` table, then assigns that table to event 5 of the `F1_PhMod` row. If the source sequence uses older `2P` row metadata, the event-assignment step upgrades that row to `3P` so TNMR does not need to convert it on load.

In [ ]:
waveform = Sine(
    points=32,
    phase_min=0.0,
    phase_max=180.0,
    amp_min=None,
    amp_max=None,
    phase_steps=1024,
)
table_name = "ph1"

sequence.set_waveform_to_table(table_name, waveform, quantity="phase")
sequence.assign_table_to_event("F1_PhMod", 5, table_name)

sequence.analysis["edits"][-2:]

## 7. Save And Reload

The reload check confirms the generated TPS can be parsed again and that every referenced table has a table definition.

In [ ]:
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=True)
sequence.write_tps(OUTPUT_TPS_PATH)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
table_names = {table["name"] for table in reloaded.tables if table.get("name")}
missing = {
    reference.table_name
    for reference in reloaded.table_references()
    if reference.table_name and reference.table_name not in table_names
}
assert not missing, sorted(missing)

print("Wrote JSON:", OUTPUT_JSON_PATH)
print("Wrote TPS:", OUTPUT_TPS_PATH)
print("Reloaded table count:", len(reloaded.tables))